In [1]:

# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit


# ==========================================
# 2. PROJECT CONFIGURATION
# ==========================================

# Locate the project root, whether Jupyter is
# running from the root folder or notebooks/.

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "data" / "raw").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Make sure data/raw exists."
    )

# Define directories
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"

# Create output directories if they don't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# ==========================================
# 3. DATASET FILES
# ==========================================

DATA_FILES = [
    DATA_DIR / "CONDA_train.csv",
    DATA_DIR / "CONDA_valid.csv",
]


# ==========================================
# 4. EXPERIMENT CONFIGURATION
# ==========================================

RANDOM_STATE = 42

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15


# ==========================================
# 5. COLUMN STANDARDIZATION
# ==========================================

COLUMN_MAP = {
    "matchId": "match_id",
    "conversationId": "conversation_id",
    "playerId": "player_id",
    "playerSlot": "player_slot",
    "chatTime": "chat_time",
    "utterance": "utterance",
    "slotTokens": "slot_tokens",
    "intentClass": "intent_class",
    "slotClasses": "slot_classes",
}


# ==========================================
# 6. VALIDATION CONFIGURATION
# ==========================================

VALID_INTENTS = {"E", "I", "A", "O"}

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", DATA_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Configuration successfully loaded!")

Project root: C:\Users\celes\Documents\GitHub\197-miniproject
Raw data directory: C:\Users\celes\Documents\GitHub\197-miniproject\data\raw
Processed data directory: C:\Users\celes\Documents\GitHub\197-miniproject\data\processed
Configuration successfully loaded!


In [2]:

# ==========================================
# LOAD THE CONDA DATASET
# ==========================================

# Load the labeled training and validation files
train_df = pd.read_csv(DATA_FILES[0])
valid_df = pd.read_csv(DATA_FILES[1])

# Combine them for our new match-level split
df = pd.concat(
    [train_df, valid_df],
    ignore_index=True
)

# Standardize column names
df = df.rename(columns=COLUMN_MAP)

print("Dataset successfully loaded!")
print("----------------------------------")
print(f"Training file: {len(train_df):,} rows")
print(f"Validation file: {len(valid_df):,} rows")
print(f"Combined dataset: {len(df):,} rows")

print("\nDataset columns:")
print(df.columns.tolist())

print("\nFirst five records:")
display(df.head())

Dataset successfully loaded!
----------------------------------
Training file: 26,921 rows
Validation file: 8,974 rows
Combined dataset: 35,895 rows

Dataset columns:
['Id', 'match_id', 'conversation_id', 'utterance', 'chat_time', 'player_slot', 'player_id', 'intent_class', 'slot_classes', 'slot_tokens']

First five records:


,Id,match_id,conversation_id,utterance,chat_time,player_slot,player_id,intent_class,slot_classes,slot_tokens
0,11263,697,3193,wow!,76,0,ANTS IN MY EYES JOHNSON,O,O,"wow (O),"
1,13741,843,3809,WTF,1563,5,M.k,O,T,"WTF (T),"
2,22125,1412,6199,wpe wpe,2853,1,Acqua Ragia,O,O O,"wpe (O), wpe (O),"
3,6453,439,1875,hahaha,1038,0,juicebox,O,O,"hahaha (O),"
4,9644,601,2713,wtf,1661,5,KAIST.Shadows,O,T,"wtf (T),"


In [3]:

# ==========================================
# INITIAL DATASET INSPECTION
# ==========================================

print("Dataset Information:")
df.info()

print("\nMissing Values:")
display(df.isnull().sum())

print("\nIntent Class Distribution:")
display(
    df["intent_class"]
    .value_counts(dropna=False)
    .rename_axis("intent_class")
    .reset_index(name="count")
)

print("\nUnique Matches:", df["match_id"].nunique())
print("Unique Conversations:", df["conversation_id"].nunique())

print("\nDuplicate Records:", df.duplicated().sum())

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 35895 entries, 0 to 35894
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Id               35895 non-null  int64
 1   match_id         35895 non-null  int64
 2   conversation_id  35895 non-null  int64
 3   utterance        35887 non-null  str  
 4   chat_time        35895 non-null  int64
 5   player_slot      35895 non-null  int64
 6   player_id        35886 non-null  str  
 7   intent_class     35895 non-null  str  
 8   slot_classes     34793 non-null  str  
 9   slot_tokens      34793 non-null  str  
dtypes: int64(5), str(5)
memory usage: 2.7 MB

Missing Values:


Id                    0
match_id              0
conversation_id       0
utterance             8
chat_time             0
player_slot           0
player_id             9
intent_class          0
slot_classes       1102
slot_tokens        1102
dtype: int64


Intent Class Distribution:


,intent_class,count
0,O,26611
1,E,4711
2,A,2299
3,I,2274



Unique Matches: 1917
Unique Conversations: 10902

Duplicate Records: 0


In [4]:

# ==========================================
# INSPECT MISSING UTTERANCES
# ==========================================

missing_utterances = df[df["utterance"].isna()]

print("Records with missing utterances:")
print("Total:", len(missing_utterances))

display(
    missing_utterances[
        [
            "Id",
            "match_id",
            "conversation_id",
            "chat_time",
            "intent_class"
        ]
    ]
)

print("\nMissing player IDs:")
display(
    df[df["player_id"].isna()][
        [
            "Id",
            "match_id",
            "conversation_id",
            "utterance",
            "intent_class"
        ]
    ]
)


Records with missing utterances:
Total: 8


,Id,match_id,conversation_id,chat_time,intent_class
701,20263,1270,5633,2026,O
7720,3078,229,962,2559,O
8623,10895,676,3055,1898,O
12751,10096,626,2828,1847,O
15936,6060,424,1802,2944,O
20539,32559,2132,9154,1406,O
24594,163,19,63,-59,O
35386,2525,183,774,5287,O



Missing player IDs:


,Id,match_id,conversation_id,utterance,intent_class
2975,9730,606,2736,fuck you,E
9316,9742,606,2742,fuck you tuskar,E
11099,9740,606,2742,lol,O
12305,9748,606,2742,lost,O
16215,9732,606,2737,gay,E
18472,9735,606,2740,aw,O
22947,9741,606,2742,intro boys,O
24107,9752,606,2742,gg,O
29822,9731,606,2737,shhhh,O


In [5]:

# ==========================================
# INITIAL DATA CLEANING
# ==========================================

# Preserve the original combined dataset
df_clean = df.copy()

# Record the original number of rows
initial_rows = len(df_clean)

# Remove records with missing utterances
df_clean = df_clean.dropna(subset=["utterance"]).copy()

# Normalize Unicode and whitespace
def clean_utterance(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df_clean["utterance"] = df_clean["utterance"].apply(
    clean_utterance
)

# Remove messages that become empty after cleaning
df_clean = df_clean[
    df_clean["utterance"].str.len() > 0
].copy()

# Validate intent labels
df_clean["intent_class"] = (
    df_clean["intent_class"]
    .astype(str)
    .str.strip()
    .str.upper()
)

invalid_labels = set(df_clean["intent_class"]) - VALID_INTENTS

if invalid_labels:
    raise ValueError(
        f"Unexpected intent labels found: {invalid_labels}"
    )

# Report cleaning results
removed_rows = initial_rows - len(df_clean)

print("INITIAL CLEANING RESULTS")
print("----------------------------------")
print(f"Original records: {initial_rows:,}")
print(f"Remaining records: {len(df_clean):,}")
print(f"Removed records: {removed_rows:,}")
print(f"Missing utterances: {df_clean['utterance'].isna().sum()}")
print(f"Empty utterances: {(df_clean['utterance'].str.len() == 0).sum()}")
print(f"Missing player IDs: {df_clean['player_id'].isna().sum()}")
print(f"Invalid labels: {(~df_clean['intent_class'].isin(VALID_INTENTS)).sum()}")

INITIAL CLEANING RESULTS
----------------------------------
Original records: 35,895
Remaining records: 35,887
Removed records: 8
Missing utterances: 0
Empty utterances: 0
Missing player IDs: 9
Invalid labels: 0


In [6]:

# ==========================================
# CREATE BINARY TOXICITY LABEL
# ==========================================

label_mapping = {
    "E": 1,
    "I": 1,
    "A": 0,
    "O": 0
}

df_clean["toxicity_binary"] = (
    df_clean["intent_class"].map(label_mapping)
)

# Verify label creation
print("Original and Binary Label Distribution:")
display(
    df_clean.groupby(
        ["intent_class", "toxicity_binary"]
    ).size().reset_index(name="count")
)

print("\nBinary Toxicity Distribution:")
display(
    df_clean["toxicity_binary"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Non-toxic", 1: "Toxic"})
)

# Check for unmapped labels
assert df_clean["toxicity_binary"].notna().all()

print("\nBinary labels successfully created!")


Original and Binary Label Distribution:


,intent_class,toxicity_binary,count
0,A,0,2299
1,E,1,4711
2,I,1,2274
3,O,0,26603



Binary Toxicity Distribution:


toxicity_binary
Non-toxic    28902
Toxic         6985
Name: count, dtype: int64


Binary labels successfully created!


In [7]:

# ==========================================
# CHRONOLOGICAL SORTING
# ==========================================

df_clean = df_clean.sort_values(
    by=[
        "match_id",
        "conversation_id",
        "chat_time",
        "Id"
    ],
    kind="stable"
).reset_index(drop=True)

print("Dataset sorted chronologically!")

display(
    df_clean[
        [
            "match_id",
            "conversation_id",
            "chat_time",
            "utterance",
            "intent_class",
            "toxicity_binary"
        ]
    ].head(10)
)


Dataset sorted chronologically!


,match_id,conversation_id,chat_time,utterance,intent_class,toxicity_binary
0,0,0,-8,force it,A,0
1,0,0,5,space created [SEPA] hah,O,0
2,0,0,9,ez 500,I,1
3,0,1,934,mvp ulti,O,0
4,0,2,1488,hah,O,0
5,0,3,1721,ka bu tooooooooooooo,O,0
6,0,4,1854,wtf [SEPA] TA? [SEPA] u srsly?,E,1
7,0,4,1882,sad spec [SEPA] noes cape for him,O,0
8,0,4,1888,wat [SEPA] that one i cant even run,O,0
9,0,4,1905,why alyway hit me [SEPA] what did i do,O,0


In [8]:

# ==========================================
# MATCH-LEVEL DATASET SPLITTING
# ==========================================

# First split: 70% training, 30% temporary
splitter_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=RANDOM_STATE
)

train_idx, temp_idx = next(
    splitter_1.split(
        df_clean,
        groups=df_clean["match_id"]
    )
)

train_df = df_clean.iloc[train_idx].copy()
temp_df = df_clean.iloc[temp_idx].copy()


# Second split: divide remaining 30% equally
splitter_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=RANDOM_STATE
)

val_idx, test_idx = next(
    splitter_2.split(
        temp_df,
        groups=temp_df["match_id"]
    )
)

valid_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()


# Sort each partition consistently
sort_columns = [
    "match_id",
    "conversation_id",
    "chat_time",
    "Id"
]

train_df = train_df.sort_values(
    sort_columns
).reset_index(drop=True)

valid_df = valid_df.sort_values(
    sort_columns
).reset_index(drop=True)

test_df = test_df.sort_values(
    sort_columns
).reset_index(drop=True)


print("MATCH-LEVEL SPLITTING COMPLETED!")


MATCH-LEVEL SPLITTING COMPLETED!


In [9]:

# ==========================================
# VERIFY DATASET PARTITIONS
# ==========================================

# Get unique match IDs from each partition
train_matches = set(train_df["match_id"])
valid_matches = set(valid_df["match_id"])
test_matches = set(test_df["match_id"])

# Check for overlapping matches
train_valid_overlap = train_matches & valid_matches
train_test_overlap = train_matches & test_matches
valid_test_overlap = valid_matches & test_matches

print("DATA LEAKAGE CHECK")
print("----------------------------------")

print("Train-Validation overlap:", len(train_valid_overlap))
print("Train-Test overlap:", len(train_test_overlap))
print("Validation-Test overlap:", len(valid_test_overlap))

# Assert that no matches overlap
assert len(train_valid_overlap) == 0
assert len(train_test_overlap) == 0
assert len(valid_test_overlap) == 0

print("\nNo match-level data leakage detected!")


# Display partition statistics
print("\nPARTITION STATISTICS")
print("----------------------------------")

for name, partition in [
    ("Train", train_df),
    ("Validation", valid_df),
    ("Test", test_df)
]:
    print(f"\n{name}:")
    print(f"  Messages: {len(partition):,}")
    print(f"  Matches: {partition['match_id'].nunique():,}")
    print(f"  Conversations: {partition['conversation_id'].nunique():,}")
    print(f"  Toxic messages: {partition['toxicity_binary'].sum():,}")
    print(
        f"  Toxicity rate: "
        f"{partition['toxicity_binary'].mean() * 100:.2f}%"
    )

print("\nTotal messages:", len(train_df) + len(valid_df) + len(test_df))
print("Original cleaned messages:", len(df_clean))

assert (
    len(train_df) + len(valid_df) + len(test_df)
    == len(df_clean)
)

print("\nAll partition checks passed!")

DATA LEAKAGE CHECK
----------------------------------
Train-Validation overlap: 0
Train-Test overlap: 0
Validation-Test overlap: 0

No match-level data leakage detected!

PARTITION STATISTICS
----------------------------------

Train:
  Messages: 25,297
  Matches: 1,341
  Conversations: 7,667
  Toxic messages: 5,037
  Toxicity rate: 19.91%

Validation:
  Messages: 5,165
  Matches: 288
  Conversations: 1,537
  Toxic messages: 879
  Toxicity rate: 17.02%

Test:
  Messages: 5,425
  Matches: 288
  Conversations: 1,697
  Toxic messages: 1,069
  Toxicity rate: 19.71%

Total messages: 35887
Original cleaned messages: 35887

All partition checks passed!


In [10]:

# ==========================================
# CLASS DISTRIBUTION PER PARTITION
# ==========================================

for name, partition in [
    ("Train", train_df),
    ("Validation", valid_df),
    ("Test", test_df)
]:
    print(f"\n{name} Class Distribution:")

    distribution = (
        partition["intent_class"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print(distribution)

# Binary toxicity distribution
binary_distribution = pd.DataFrame({
    name: partition["toxicity_binary"].value_counts(
        normalize=True
    ).mul(100)
    for name, partition in [
        ("Train", train_df),
        ("Validation", valid_df),
        ("Test", test_df)
    ]
}).fillna(0).round(2)

binary_distribution.index = [
    "Non-toxic" if i == 0 else "Toxic"
    for i in binary_distribution.index
]

print("\nBINARY TOXICITY DISTRIBUTION (%)")
display(binary_distribution)



Train Class Distribution:
intent_class
O    73.84
E    13.50
I     6.42
A     6.25
Name: proportion, dtype: float64

Validation Class Distribution:
intent_class
O    74.73
E    11.50
A     8.25
I     5.52
Name: proportion, dtype: float64

Test Class Distribution:
intent_class
O    74.89
E    12.96
I     6.75
A     5.40
Name: proportion, dtype: float64

BINARY TOXICITY DISTRIBUTION (%)


,Train,Validation,Test
Non-toxic,80.09,82.98,80.29
Toxic,19.91,17.02,19.71


In [11]:

# ==========================================
# FINAL DATA INTEGRITY CHECK
# ==========================================

print("ORIGINAL MESSAGE ID CHECK")
print("----------------------------------")

print("Duplicate IDs in combined dataset:", df["Id"].duplicated().sum())

print("\nPARTITION MESSAGE ID CHECK")

train_ids = set(train_df["Id"])
valid_ids = set(valid_df["Id"])
test_ids = set(test_df["Id"])

print("Train-Validation:", len(train_ids & valid_ids))
print("Train-Test:", len(train_ids & test_ids))
print("Validation-Test:", len(valid_ids & test_ids))

assert df["Id"].is_unique
assert not (train_ids & valid_ids)
assert not (train_ids & test_ids)
assert not (valid_ids & test_ids)

print("\nAll message ID checks passed!")

ORIGINAL MESSAGE ID CHECK
----------------------------------
Duplicate IDs in combined dataset: 0

PARTITION MESSAGE ID CHECK
Train-Validation: 0
Train-Test: 0
Validation-Test: 0

All message ID checks passed!


In [12]:

# ==========================================
# SAVE PROCESSED DATASETS
# ==========================================

# Save each partition
train_df.to_csv(
    PROCESSED_DIR / "train.csv",
    index=False
)

valid_df.to_csv(
    PROCESSED_DIR / "validation.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "test.csv",
    index=False
)

print("PROCESSED DATASETS SAVED!")
print("----------------------------------")

print("Train:", PROCESSED_DIR / "train.csv")
print("Validation:", PROCESSED_DIR / "validation.csv")
print("Test:", PROCESSED_DIR / "test.csv")

PROCESSED DATASETS SAVED!
----------------------------------
Train: C:\Users\celes\Documents\GitHub\197-miniproject\data\processed\train.csv
Validation: C:\Users\celes\Documents\GitHub\197-miniproject\data\processed\validation.csv
Test: C:\Users\celes\Documents\GitHub\197-miniproject\data\processed\test.csv


In [13]:

# ==========================================
# GENERATE DATASET SUMMARY
# ==========================================

partitions = {
    "Train": train_df,
    "Validation": valid_df,
    "Test": test_df
}

summary = []

for name, partition in partitions.items():

    summary.append({
        "partition": name,
        "total_messages": len(partition),
        "unique_matches": partition["match_id"].nunique(),
        "unique_conversations": partition["conversation_id"].nunique(),
        "toxic_messages": int(partition["toxicity_binary"].sum()),
        "non_toxic_messages": int(
            (partition["toxicity_binary"] == 0).sum()
        ),
        "toxicity_rate": round(
            partition["toxicity_binary"].mean() * 100, 2
        ),
        "E_count": int(
            (partition["intent_class"] == "E").sum()
        ),
        "I_count": int(
            (partition["intent_class"] == "I").sum()
        ),
        "A_count": int(
            (partition["intent_class"] == "A").sum()
        ),
        "O_count": int(
            (partition["intent_class"] == "O").sum()
        )
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

# Save summary
summary_df.to_csv(
    PROCESSED_DIR / "dataset_summary.csv",
    index=False
)

print("Dataset summary saved successfully!")

,partition,total_messages,unique_matches,unique_conversations,toxic_messages,non_toxic_messages,toxicity_rate,E_count,I_count,A_count,O_count
0,Train,25297,1341,7667,5037,20260,19.91,3414,1623,1580,18680
1,Validation,5165,288,1537,879,4286,17.02,594,285,426,3860
2,Test,5425,288,1697,1069,4356,19.71,703,366,293,4063


Dataset summary saved successfully!


In [14]:

# ==========================================
# VERIFY SAVED DATASETS
# ==========================================

saved_train = pd.read_csv(PROCESSED_DIR / "train.csv")
saved_valid = pd.read_csv(PROCESSED_DIR / "validation.csv")
saved_test = pd.read_csv(PROCESSED_DIR / "test.csv")

print("SAVED DATASET VERIFICATION")
print("----------------------------------")

print("Train:", len(saved_train))
print("Validation:", len(saved_valid))
print("Test:", len(saved_test))

assert len(saved_train) == len(train_df)
assert len(saved_valid) == len(valid_df)
assert len(saved_test) == len(test_df)

assert saved_train["toxicity_binary"].notna().all()
assert saved_valid["toxicity_binary"].notna().all()
assert saved_test["toxicity_binary"].notna().all()

print("\nAll saved datasets verified successfully!")

SAVED DATASET VERIFICATION
----------------------------------
Train: 25297
Validation: 5165
Test: 5425

All saved datasets verified successfully!
